In [48]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data import random_split
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
import random
import matplotlib.pyplot as plt
import numpy as np

In [49]:
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

set_seed(SEED)

# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # wymagane przez niektóre operacje CUDA
# device = torch.device("cuda")
device = torch.device("cpu")

# Transformacje


In [50]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

In [51]:
dataset = torchvision.datasets.ImageFolder(
    root="train/",
    transform=transform
)

In [52]:
class_to_idx = dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

print(class_to_idx)

{'acoustic': 0, 'antenna': 1, 'bacteria': 2, 'battery': 3, 'bean': 4, 'beetle': 5, 'bicycle': 6, 'birch': 7, 'bird': 8, 'bomb': 9, 'bread': 10, 'bridge': 11, 'camera': 12, 'carbon': 13, 'cat': 14, 'corn': 15, 'crab': 16, 'crocodilian': 17, 'echinoderm': 18, 'egg': 19, 'elephant': 20, 'fish': 21, 'flower': 22, 'frog': 23, 'fungus': 24, 'gauge': 25, 'hammer': 26, 'icecream': 27, 'kangaroo': 28, 'memorial': 29, 'monkey': 30, 'motor': 31, 'nest': 32, 'palm': 33, 'pizza': 34, 'pot': 35, 'printer': 36, 'saw': 37, 'snake': 38, 'spice': 39, 'spider': 40, 'spoon': 41, 'squash': 42, 'swine': 43, 'tea': 44, 'tomato': 45, 'towel': 46, 'truck': 47, 'turtle': 48, 'worm': 49}


In [53]:
number_of_classes = len(class_to_idx)

In [54]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_data, val_data = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

In [55]:
for images, labels in train_loader:
    print(images.shape)  # e.g. [32, 3, 224, 224]
    print(labels.shape)  # e.g. [32]
    break

torch.Size([32, 3, 64, 64])
torch.Size([32])


In [56]:
# all_samples = torch.stack([sample[0] for sample in dataset])
# print(f"Data norm: mean={all_samples.mean():.4f} | std={all_samples.std():.4f}")
# print(f"Data norm per channel: mean={all_samples.mean(axis=(0,2,3))} | std={all_samples.std(axis=(0,2,3))}")

In [57]:
def Get_Weights(weight_classes : bool = True):
    if not weight_classes:
        return torch.ones((1))
    # targets = list of class indices for each image
    counts = Counter(dataset.targets)

    # Map back to class names
    class_counts = {dataset.classes[i]: counts[i] for i in range(len(dataset.classes))}

    min_class = min(class_counts, key=class_counts.get)
    max_class = max(class_counts, key=class_counts.get)

    ## bread ma 1108, carbon ma 503, reszta po 1800

    weights = torch.tensor(list(class_counts.values()), dtype =float)
    weights = torch.reciprocal(weights)

    return weights

In [58]:
Get_Weights()

tensor([0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006,
        0.0006, 0.0009, 0.0006, 0.0006, 0.0020, 0.0006, 0.0006, 0.0006, 0.0006,
        0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006,
        0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006,
        0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006,
        0.0006, 0.0006, 0.0006, 0.0006, 0.0006], dtype=torch.float64)

In [62]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ## Warstwa konwolucyjna
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5, stride=1, padding=0)
        ## Warstwa max pooling
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.pool2 = nn.MaxPool2d(2)
        # !!!
        self.fc1 = nn.LazyLinear(120)
        # self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, number_of_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        # !!!
        x = torch.flatten(x, start_dim=1) # flatten all dimensions except batch
        # print(x.size())
        # !!!
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [63]:
import torch.optim as optim

set_seed(42)

criterion = nn.CrossEntropyLoss()
net = Net().to(device)
optimizer = optim.Adam(net.parameters(), lr=0.001)

net

Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): LazyLinear(in_features=0, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=50, bias=True)
)

In [64]:
for epoch in range(5):  # loop over the dataset multiple times

    running_loss = 0.0
    for data in train_loader:
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()

    print('[%d/5] loss: %.3f' %
          (epoch+1 ,  running_loss / 2000))
    running_loss = 0.0

print('Finished Training')

[1/5] loss: 3.610
[2/5] loss: 3.215
[3/5] loss: 3.058
[4/5] loss: 2.941
[5/5] loss: 2.848
Finished Training
